# 04. 노이즈 채널(01) + 태거(02) 앙상블
01 빔 서치 점수에 태거 글자 확률을 더한다 (재학습 없음).

`score = 채널 + lm_weight·LM + dict_weight·사전 + tagger_weight·(log P태거 초+중+종)`

- 01 = 받침·연음·사전에 강함 / 태거 = 넓은 문맥(뉴스↔누수)에 강함
- 출발 파라미터는 `outputs/best_params.json`(01 sweep 최고) 에서 읽고, `--grid` 로 준 값만 바꿔 탐색
- `tagger_weight 0` = 01 단독 (0.9834 재현되면 정상)
- 조합마다 `outputs/ens_runs.csv` 기록, `best_ensemble.json` 은 매 조합 갱신(중간에 끊겨도 남음)

**전제:** 01 sweep, 02 학습 완료. **GPU 런타임**(태거 추론) — 빔 서치는 CPU 라서 20조합 약 20~30분.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/데이콘/난독화
!ls -la outputs/tagger_best.pt outputs/best_params.json
!nproc

## 1) 빠른 동작 확인 (val 100행, 2조합)

In [ ]:
!python ensemble.py --mode val --val_limit 100 --grid 'tagger_weight=0,1.0'

## 2) 탐색 (검증 전체)
기본: `tagger_weight 0,0.3,0.6,1.0,1.5 × lm_weight 0.9,1.1 × dict_oov 0.25,0.5` = 20조합

In [ ]:
!python ensemble.py --mode val

In [ ]:
import pandas as pd
r = pd.read_csv('outputs/ens_runs.csv')
r = r[r.n_val == r.n_val.max()]
r.sort_values('f1', ascending=False)[['tagger_weight','lm_weight','dict_oov','beam','dict_weight','f1','sec']].head(15)

## 3) 01 / 02 / 03 / 04 비교

In [ ]:
for name, f in [('01 noisy-channel', 'outputs/val_pred.csv'),
                ('02 tagger', 'outputs/val_pred_tagger.csv'),
                ('03 tagger+dict', 'outputs/val_pred_tagger_dict.csv'),
                ('04 ensemble', 'outputs/val_pred_ensemble.csv')]:
    try:
        d = pd.read_csv(f); print(f'{name:18s} F1 {d.f1.mean():.4f}  (rows {len(d)})')
    except FileNotFoundError:
        print(f'{name:18s} 파일 없음')

## 4) test 예측 → `outputs/submission_ensemble.csv`

In [ ]:
!python ensemble.py --mode test
pd.read_csv('outputs/submission_ensemble.csv').head()